## **Semantic Chunking**

In [1]:
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

/mnt/Linux/Projects/grag/grag_core/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL = "onnx_models/embeddinggemma_300m/onnx/model.onnx"
TOKENIZER = "onnx_models/embeddinggemma_300m"

class ONNXEmbedder:
    def __init__(self, model: str = MODEL, tokenizer: str = TOKENIZER, max_length: int = 1024):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer)
        self.model = ort.InferenceSession(model, providers=["CPUExecutionProvider"])
        self.max_length = max_length

    def _get_model_input_names(self) -> set[str]:
        return {inp.name for inp in self.model.get_inputs()}

    def _build_feed(self, inputs: dict) -> dict:
        feed = dict(inputs)
        required = self._get_model_input_names()

        if "position_ids" in required:
            input_ids = feed["input_ids"]
            feed["position_ids"] = np.arange(input_ids.shape[1], dtype=np.int64)[None].repeat(input_ids.shape[0], axis=0)

        # Collect past_key_value input specs and zero-fill them
        kv_inputs = {inp.name: inp for inp in self.model.get_inputs() if inp.name.startswith("past_key_values")}
        for name, spec in kv_inputs.items():
            # Shape is [batch, heads, 0, head_dim] for empty KV cache; use spec shape hints
            shape = [d if isinstance(d, int) and d >= 0 else 0 for d in spec.shape]
            shape[0] = feed["input_ids"].shape[0]  # batch size
            dtype = np.float16 if spec.type == "tensor(float16)" else np.float32
            feed[name] = np.zeros(shape, dtype=dtype)

        return feed

    def encode(
        self,
        texts: list[str],
        batch_size: int = 32,
        normalize: bool = True,
    ) -> np.ndarray:
        all_embeddings: list[np.ndarray] = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="np",
            )
            feed = self._build_feed(dict(inputs))
            outputs = self.model.run(None, feed)
            emb = outputs[0]  # (batch, seq_len, hidden) — last hidden state
            # Mean pool over sequence dimension if 3D (batch, seq_len, hidden)
            if emb.ndim == 3:
                mask = feed["attention_mask"].astype(np.float32)[..., None]
                emb = (emb * mask).sum(axis=1) / mask.sum(axis=1).clip(min=1e-9)

            if normalize:
                emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)

            all_embeddings.append(emb)

        return np.concatenate(all_embeddings, axis=0)

In [3]:
def read_markdown_file(file_path: str) -> str:
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

In [ ]:
document = read_markdown_file("doc_sample.md")

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
texts = text_splitter.split_text(document)

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

model = ONNXEmbedder()

def semantic_chunking(sentences: list[str], threshold: float = 0.7) -> list[str]:

    embeddings=model.encode(sentences)

    # Initialize parameters
    threshold = threshold  # control chunk tightness
    chunks = []
    current_chunk=[sentences[0]]

    ## Semantic grouping based on threshold

    for i in range(1, len(sentences)):
        sim = cosine_similarity(
            [embeddings[i - 1]],
            [embeddings[i]]
        )[0][0]

        if sim>=threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk=[sentences[i]]

    # Append the last chunk
    chunks.append(" ".join(current_chunk))

    return chunks

In [ ]:
semantic_chunks = semantic_chunking(texts, threshold=0.7)

In [ ]:
from langchain_core.documents import Document

lc_document_loader = [Document(page_content=chunk) for chunk in semantic_chunks]

In [ ]:
semantic_chunks[5:10]

['addition to this, we can also run multiple workers in the',
 '| Model | Configuration | processed tokens/s (purple) | generated token/s (green) |',
 '|---|---|---|---|\n| bigcode/starcoderbase-3b | without bud engine | 63.29 | 9.31 | | bigcode/starcoderbase-3b | with bud engine | 992.36 | 167.58 | | bigcode/starcoderbase-3b | with bud engine multiworker | 3969.44 | 670.32 | | codellama/CodeLlama-7b-hf | without bud engine multiworker | 32.17 | 4.36 | | codellama/CodeLlama-7b-hf | with bud engine | 593.74 | 93.69 | | codellama/CodeLlama-7b-hf | with bud engine multiworker | 1852.32 | 305.3 |',
 'Figure 1: Improvement of token/s with the use of Bud Inference engine with 32 vCPU on 4th Gen',
 'Intel® Xeon® Scalable Processors']

In [ ]:
texts[12:22]

['addition to this, we can also run multiple workers in the',
 '| Model | Configuration | processed tokens/s (purple) | generated token/s (green) |',
 '|---|---|---|---|\n| bigcode/starcoderbase-3b | without bud engine | 63.29 | 9.31 |',
 '| bigcode/starcoderbase-3b | with bud engine | 992.36 | 167.58 |',
 '| bigcode/starcoderbase-3b | with bud engine multiworker | 3969.44 | 670.32 |',
 '| codellama/CodeLlama-7b-hf | without bud engine multiworker | 32.17 | 4.36 |',
 '| codellama/CodeLlama-7b-hf | with bud engine | 593.74 | 93.69 |',
 '| codellama/CodeLlama-7b-hf | with bud engine multiworker | 1852.32 | 305.3 |',
 'Figure 1: Improvement of token/s with the use of Bud Inference engine with 32 vCPU on 4th Gen',
 'Intel® Xeon® Scalable Processors']